In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

import psutil
import math
import gc
from collections import Counter
import re


import os, random
import posixpath
from pyhdas.frequency import spectrogram, add_db, energy, power_spectrum
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta

In [3]:
import warnings

# Suppress FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
def calculate_low_frequency_spectrum(
    strain,
    stride=16,
    low_frequency_cutoff=50,
    low_frequency_min=0.1,
):
    """Calculate low-frequency spectrum from strain signal

    Parameters
    ----------
    strain : xarray
        DAS strain measurement time series
    stride: integer, number of positions to calculate at once
        Using a higher stride will be faster on computers with a lot of RAM.
    low_frequency_cutoff : float, default 50
        Cutoff of the frequencies to discard when saving the low frequency
        outputs.
    low_frequency_min : float, default 0.1
        Minimal frequency for the power_spectrum of the low
        frequencies output

    Returns
    -------
    xarray
        low-frequency spectrum
    """

    all_spectra = []
    for j in range(0, len(strain.position), stride):
        low_freq_spectrum_ = power_spectrum(
            strain.isel(position=slice(j, j + stride)),
            min_frequency=low_frequency_min,
            keep_time_dim=True,
        )
        low_freq_spectrum_ = low_freq_spectrum_.sel(
            freq=slice(
                low_frequency_cutoff,
            )
        )
        all_spectra.append(low_freq_spectrum_)

    low_freq_spectrum = xr.concat(all_spectra, dim="position")

    return low_freq_spectrum

In [5]:
# select 200 random files from the directory
normal_data_dir = "data/19"
random.seed(58)
files = random.sample(os.listdir(normal_data_dir), 100)

In [11]:
def normal_low():

    all_rows = []

    for file in files:

        filepath = [posixpath.join(normal_data_dir, file)]

        # open file one by one
        ds_raw = concat_raw_data(filepath)

        # select locations 4210-4260
        poi = np.arange(5800, 5900, 10)

        ds_raw = ds_raw.sel(position=poi)
        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)
        
        fig,ax = plt.subplots(figsize=(15,6))
        ds_lowfreq_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
        ax.set_title(f"lowfrequency position spectrogram")
        
        # save in the normal data directory
        output_folder = Path(f"plots/new_normal_low_freq_spectrograms")
        output_folder.mkdir(parents=True, exist_ok=True)

        plot_filename = f"{file[:-4]}_high_FP.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')

        plt.close(fig)

        # row = dict()
        # row["start"] = file[:-4]
        # row["end"] = None
        # row["event_label"] = "normal"

        # all_rows.append(row)
        
        print(f"Done with {file}")
        del ds_raw, ds_lowfreq_spect
        gc.collect()

    # if all_rows:
    #     final_df = pd.DataFrame(all_rows)
    #     output_folder = Path("low_freq_data_csv")
    #     output_folder.mkdir(parents=True, exist_ok=True)
    #     final_df.to_csv(output_folder / "annotations_normal_hours.csv", index=False)


In [12]:
normal_low()

Done with 2021_02_19_20h39m41s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_06h43m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_07h01m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_22h44m41s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_06h51m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_01h24m38s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_06h34m39s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_16h10m40s_HDAS_2DRawData_Strain.bin
Done with 2021_02_19_14h20m10s_HDAS_2DRawData_Strain.bin


SystemError: CPUDispatcher(<function denoise_flagged at 0x7fbd1903a0d0>) returned a result with an error set

In [ ]:
# load the event table 
events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])

In [ ]:
def event_level():
    # loop through event table 
    for _, event in events.iterrows():
        # load the event data
        start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]
        # select the locations: i chose the ones that are not in the poi list
        poi = np.arange(4200, 4310, 10)
        # if the duration of the event is more than 2 minutes, print the event_label, date and start and end time, and move on to the next row
        event_duration = (end - start).total_seconds()
        if event_duration > 120 or event_duration <= 1:
            continue
    
        # load the strain data based on the start and the end of the event
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)
        start = start.tz_localize("UTC")
        end = end.tz_localize("UTC")
        day = start.day

        dir_data = Path(fr"data/{day}")
        file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        ds_raw = concat_raw_data(file_list)
        start = start.tz_localize(None)
        end = end.tz_localize(None)

        # sound level 
        ds_raw = ds_raw.sel(time=slice(start, end), position=poi)
        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)

        fig,ax = plt.subplots(figsize=(15,6))
        ds_lowfreq_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
        ax.set_title(f"lowfrequency position spectrogram")

        # save the plot
        output_folder = Path(f"plots/low_freq_plots/100_Hz/{label}")
        output_folder.mkdir(parents=True, exist_ok=True)
  
        plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_low_freq.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')

        plt.close(fig)
        
        print(f"Done with {start}, {end}, {label}")
        del ds_raw, ds_lowfreq_spect
        gc.collect()

In [ ]:
def event_level_csv():
    all_rows = []
    label_counter = Counter()

    for _, event in events.iterrows():
        start, end, event_label = event["start"], event["end"], event["label_anon"]

        # Skip events that are too short or too long
        event_duration = (end - start).total_seconds()
        if event_duration > 120 or event_duration <= 1:
            continue

        # Prepare timestamps and load data
        start_utc = pd.Timestamp(start).tz_localize("UTC")
        end_utc = pd.Timestamp(end).tz_localize("UTC")
        day = start_utc.day

        dir_data = Path(fr"data/{day}")
        file_list = list(aragon_select_files(dir_data, start_utc, end_utc, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {event_label}")
            continue

        ds_raw = concat_raw_data(file_list)

        # Remove timezone info for slicing
        start = start_utc.tz_localize(None)
        end = end_utc.tz_localize(None)

        # Focus on position 4240
        poi = [4240]
        ds_raw = ds_raw.sel(time=slice(start, end), position=poi)
        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)

        pxx = ds_lowfreq_spect.Pxx_dB.sel(position=4240).values  

        # Flatten and create column names
        flat_values = pxx.flatten()
        col_names = [f"f{fi}" for fi in range(len(flat_values))]

        # Build a row dict
        row = dict(zip(col_names, flat_values))
        row["start"] = start
        row["end"] = end
        row["event_label"] = event_label
        
        # Locate corresponding plot file
        plots_dir = Path(f"plots/low_freq_plots/{event_label}")
        file_pattern = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_low_freq"
        
        matched_file = next((f for f in plots_dir.glob(f"{file_pattern}_*.png")), None)

        if matched_file:
            # Extract label: last character before .png
            match = re.search(r"_low_freq_([01\?])\.png$", matched_file.name)
            label = match.group(1) if match else "?"
        else:
            print(f"No matching plot file for event {event_label} from {start} to {end}")
            label = "?"

        row["label"] = label
        label_counter[label] += 1

        all_rows.append(row)

        print(f"Processed: {start}, {end}, {event_label}")
        del ds_raw, ds_lowfreq_spect, pxx
        gc.collect()

    if all_rows:
        final_df = pd.DataFrame(all_rows)
        output_folder = Path("low_freq_data_csv")
        output_folder.mkdir(parents=True, exist_ok=True)
        final_df.to_csv(output_folder / "low_freq_flattened_4240.csv", index=False)
        print(f"\nCSV saved with shape {final_df.shape}")
        print(f"Label counts: {dict(label_counter)}")
    else:
        print("No data to save.")


In [ ]:
event_level_csv()

In [ ]:

for row, event in events.iterrows(): 

    start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    # Skip events that are too short or too long
    event_duration = (end - start).total_seconds()
    if event_duration > 300 or event_duration <= 1:
        continue
    
    start_day = start.day

    end = end.tz_localize("UTC")
    start = start.tz_localize("UTC")

    end_file = start
    dir_data = Path(fr"data/{end.day}")


    while end_file < end:

        end_file = start + pd.Timedelta(seconds=60)

        print(start, end_file)

        # load the file based on the time provided
        file_list = list(aragon_select_files(dir_data, start, end_file, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        # extract the sound levels 
        ds_raw = concat_raw_data(file_list)

        start = start.tz_localize(None)
        end_file = end_file.tz_localize(None)

        pipe_locs = np.arange(4200, 4310, 10)

        current_length = len(ds_raw.sel(time=slice(start, end_file), position=pipe_locs).time)

        # in case tehre are not enough data points, pad 
        if current_length < 120000:
            milliseconds_to_pad = int(math.ceil((120000 - current_length)/2))
            ds_raw = ds_raw.sel(time=slice(start, end_file+pd.Timedelta(milliseconds=milliseconds_to_pad)), position=pipe_locs)
        else:
            ds_raw = ds_raw.sel(time=slice(start, end_file), position=pipe_locs)
        

        ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw)


        fig,ax = plt.subplots(figsize=(15,6))
        ds_lowfreq_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
        ax.set_title(f"lowfrequency position spectrogram")

        # save the plot
        output_folder = Path(f"plots/new_low_freq_plots/{label}")
        output_folder.mkdir(parents=True, exist_ok=True)
  
        plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end_file.strftime('%H:%M:%S')}_low_freq.png"
        plt.savefig(output_folder / plot_filename, bbox_inches='tight')
        plt.close(fig)


        end_file = end_file.tz_localize("UTC")
        start = end_file

        del ds_raw, ds_lowfreq_spect
        gc.collect()

    print(f"Done with row {row}")

In [ ]:
all_rows = []

for row, event in events.iterrows(): 

    start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    # Skip events that are too short or too long
    event_duration = (end - start).total_seconds()
    if event_duration > 300 or event_duration <= 1:
        continue

    end = end.tz_localize("UTC")
    start = start.tz_localize("UTC")

    end_file = start
    dir_data = Path(fr"data/{end.day}")


    while end_file < end:

        end_file = start + pd.Timedelta(seconds=60)

        print(start, end_file)
        row = dict()
        row["start"] = start
        row["end"] = end_file
        row["event_label"] = label
        row["0"] = []
        row["1"] = []

        all_rows.append(row)
        
        start = end_file


if all_rows:
        final_df = pd.DataFrame(all_rows)
        output_folder = Path("low_freq_data_csv")
        output_folder.mkdir(parents=True, exist_ok=True)
        final_df.to_csv(output_folder / "annotations.csv", index=False)

In [16]:
def get_low_freq_data(csv_directory):

    output_folder = Path("low_freq_data")
    output_folder.mkdir(parents=True, exist_ok=True)

    position_table = pd.read_csv('position_table.csv')

    positions = [1550, 1600]
    selected = position_table[(position_table['location_name']=='quiet') & (~position_table['position_fiber'].isin(positions))]
    locs = selected['position_fiber'].values

    output_file = Path("low_freq_data/low_freq_test_normal_day.csv")


    for annotation_file in os.listdir(csv_directory):

        if annotation_file != "annotations_normal_hours.csv":
            continue
        
        csv_path = os.path.join(csv_directory, annotation_file)
        annotations_df = pd.read_csv(csv_path)

        print(f"Working with file: {annotation_file}")

        for idx, row in annotations_df.iterrows():

            if idx < 3:
                continue

            if pd.notna(row['end']):
                start = pd.Timestamp(row['start'])
                end = pd.Timestamp(row['end'])
                day = start.day

                dir_data = Path(fr"data/{day}")
                file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))

                start = start.tz_localize(None)
                end = end.tz_localize(None)
            else:
                dir_data = Path(fr"data/19")
                file_list = [posixpath.join(dir_data, f"{row['start']}.bin")]

            try:
                ds_raw = concat_raw_data(file_list)
            except Exception as e:
                print(f"Failed to load raw data for row {idx}: {e}")
                continue

            for col in locs: #annotations_df.columns[3:]:  # location columns start from 4th column
                cell_value = 0 #row[col]

                if cell_value in [0, 1]:
                    location = [int(col)]

                    # sound level 
                    if len(file_list) > 1:
                        ds_raw_slice = ds_raw.sel(time=slice(start, end), position=location)
                    else:
                        ds_raw_slice = ds_raw.sel(position=location)

                    ds_lowfreq_spect = calculate_low_frequency_spectrum(ds_raw_slice)

                    pxx = ds_lowfreq_spect.Pxx_dB.values  

                    # Flatten and create column names
                    flat_values = pxx.flatten()
                    col_names = [f"f{fi}" for fi in range(len(flat_values))]

                    # Build a row dict
                    r = dict(zip(col_names, flat_values))
                    r["start"] = row['start']
                    r["end"] = row['end']
                    r["location"] = location
                    r["event_label"] = row['event_label']
                    r["label"] = cell_value

                    row_df = pd.DataFrame([r])

                    # Check if file already exists to manage header
                    write_header = not output_file.exists()

                    # Save row to CSV
                    row_df.to_csv(output_file, mode='a', index=False, header=write_header)
                    del ds_raw_slice, ds_lowfreq_spect

            del ds_raw
            gc.collect()
                        
            print(f"Done with row {idx}")
        


In [ ]:
get_low_freq_data('low_freq_data_csv/')

Working with file: annotations_normal_hours.csv


Done with row 3
Done with row 4
Done with row 5
Done with row 6
Done with row 7
Done with row 8
Done with row 9
Done with row 10
Done with row 11
Done with row 12
Done with row 13
Done with row 14
Done with row 15
Done with row 16
Done with row 17
Done with row 18
Done with row 19
Done with row 20
Done with row 21
Done with row 22
Done with row 23
Done with row 24
Done with row 25
Done with row 26
Done with row 27
Done with row 28
Done with row 29
Done with row 30
Done with row 31
Done with row 32
Done with row 33
Done with row 34
Done with row 35
Done with row 36
Done with row 37
Done with row 38
Done with row 39
Done with row 40
Done with row 41
Done with row 42
Done with row 43
Done with row 44
Done with row 45
Done with row 46
Done with row 47
Done with row 48
Done with row 49
Done with row 50
Done with row 51
Done with row 52
Done with row 53
Done with row 54
Done with row 55
Done with row 56
Done with row 57
Done with row 58
Done with row 59
Done with row 60
Done with row 61
Done

In [18]:
position_table = pd.read_csv('position_table.csv')

positions = [1550, 1600]
selected = position_table[(position_table['location_name']=='quiet') & (~position_table['position_fiber'].isin(positions))]

In [20]:
len(selected['position_fiber'].values)

469